# Evaluating the Models on the Test Set

This notebook extracts GIST descriptors for `dataset/test/*.png` (if needed) and runs batched `torch.no_grad()` inference for all four models:
- Raw MLP
- GIST MLP
- Raw RNN
- GIST RNN

In [ ]:
import sys
from pathlib import Path

ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
if str(ROOT / "src") not in sys.path:
    sys.path.insert(0, str(ROOT / "src"))

import cv2
import numpy as np
import pandas as pd
import torch

from image_classifier.config import ProjectPaths, TrainConfig
from image_classifier.data import gist
from image_classifier.engine import get_device

paths = ProjectPaths(root=ROOT)
cfg = TrainConfig()
device = get_device()
print(f"PyTorch {torch.__version__} | Device: {device}")

### 1. Extract GIST Features for Test Images

In [ ]:
paths.gist_test_dir.mkdir(parents=True, exist_ok=True)
test_images = sorted(paths.test_images_dir.glob("*.png"))
for img_path in test_images:
    img = cv2.imread(str(img_path), cv2.IMREAD_GRAYSCALE)
    if img is None:
        raise FileNotFoundError(f"Could not read {img_path}")
    feat = gist(img, image_size=cfg.image_size)
    np.save(paths.gist_test_dir / f"{img_path.name}.npy", feat)

print(f"Prepared {len(test_images)} test GIST descriptors in {paths.gist_test_dir}")

### 2. Run Batched Inference Across All Four Models

In [ ]:
sys.path.insert(0, str(ROOT / "scripts"))
from evaluate import CHECKPOINT_MAP, predict_test_set

outputs = {}
for model_name in CHECKPOINT_MAP:
    out_csv = predict_test_set(model_name, paths, cfg, device)
    outputs[model_name] = pd.read_csv(out_csv)

outputs["gist_rnn"].head(10)